# Quantum State Validator - Project report

*Final documentation notebook. Everything below is written from the project's own
history: the executed notebooks 01-12, the session reports in `reports/`, and the git
log. Nothing is reconstructed from memory alone.*

## 1. What this project is

Quantum State Validator (QSV) started as a portfolio ML project with a simple pitch:
train a classifier to decide whether a quantum state vector is physically valid, i.e.
whether its amplitudes satisfy the normalization condition

$$\sum_{i=1}^{d} |c_i|^2 = 1.$$

It ended up as something more interesting: a case study in **when machine learning is
the right tool and when it is not**, built on an honest measurement-noise model, with
every claim backed by an executed notebook. The final deliverables are a Python
package (`pip install`, `from qsv import validate_state`), a REST API wrapping the
same logic, an interactive learning web app, and the twelve notebooks that document
how the science actually unfolded - including the mistakes.

## 2. Theoretical foundations

**States and the Born rule.** A pure state of a d-dimensional quantum system is a
complex vector $|\psi\rangle = (c_1, \dots, c_d)$. Measuring in the computational
basis yields outcome $i$ with probability $p_i = |c_i|^2 / \|\psi\|^2$ (Born rule).
For the probabilities to be a bare $|c_i|^2$, the state must be normalized:
$\|\psi\|^2 = \sum_i |c_i|^2 = 1$.

**The norm is not observable.** The measurement statistics of $|\psi\rangle$ and
$k|\psi\rangle$ are identical - the Born rule renormalizes. Checking
$\|\psi\|^2 = 1$ is therefore a *data quality* check on reported amplitude vectors
(tomography reconstructions, simulator outputs, numerical pipelines), not a physical
measurement. This reframing, forced on us by the leakage incident (section 5), is what
gave the project its identity: QSV is the model of a data-qualification subsystem, the
kind every quantum instrument carries.

**The measurement-noise model.** Real amplitude estimates carry shot noise. We model
a reconstruction from N measurement repetitions as

$$\hat{c}_i = c_i + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2),
\qquad \sigma = \frac{1}{2\sqrt{N}},$$

the universal 1/sqrt(N) counting-statistics law. A key derived fact, verified
numerically to 4 decimal places in notebook 08: the norm estimator is biased,
$\mathbb{E}[\|\hat{\psi}\|^2] = \|\psi\|^2 + 2d\sigma^2$, because each of the 2d real
components contributes $\mathbb{E}[\varepsilon^2] = \sigma^2$ inside a square. Every
decision rule in the package corrects for this bias.

**Scale-invariant vs scale-sensitive features.** Any feature computed on the
renormalized probabilities $\tilde{p}_i = p_i / \sum_j p_j$ is invariant under
$c \to kc$ and therefore carries *zero* information about validity - while any
scale-sensitive feature carries *all* of it. On exact data the problem is either
trivial or impossible; only noise makes it statistics. This dichotomy is enforced
mechanically by a unit test.

## 3. Methodological choices and why

**Experiments before interpretations.** From notebook 09 onward, every experiment was
run before a single line of interpretation was written. This rule exists because it
caught me twice in one day: I predicted the Random Forest would beat the threshold
test under correlated noise (it did not), and I drafted a conclusion saying the
threshold test "beats" the forest that the regenerated dataset then contradicted by
0.4 points (fixed to "statistically tied" before commit). Text follows numbers, never
the reverse.

**A single class boundary.** Early on, each invalid-state generation strategy had its
own notion of "close to 1". The fix (audit item F2) centralizes one rule: every
invalid state satisfies $|\|\psi\|^2 - 1| \geq$ `norm_margin` (default 0.05). The
forbidden band embodies a physical fact: a state at $\|\psi\|^2 = 1.001$ is
indistinguishable from a rounding error and has no defensible label.

**Seed discipline.** Every generator takes a seed; the shipped dataset is
`create_dataset(5000, 5000, dim=4, seed=42)` and notebook 04 is its executable
documentation - re-running it reproduces the CSV byte for byte. This paid off
concretely: a development environment was lost mid-session and the re-run produced
bit-identical results minutes later.

**Statistics first, ML where it earns its place.** The central methodological result
(milestone 4, notebooks 08-12): a well-built statistic matches or beats a 100-tree
Random Forest in every stationary regime; ML earns its place when the problem is
non-stationary (calibration drift) or diagnostic (cause classification), and performs
best hybridized with the physics. Consequently the shipped validators are statistics,
not models - an architecture decision, not a shortcut.

## 4. Architecture

```
src/qsv/                    installable package (pip install -e .)
  validators.py             pure decision logic  <- THE single source of truth
  api.py                    FastAPI service, thin wrapper over validators.py
  data_generation.py        valid/invalid state generators, F2 margin guarantee
  features.py               invariant vs sensitive features, noise models
  preparation.py            known-target preparation QA generator
  preprocessing.py          stratified splits, leak-free scaling
notebooks/01-06             implementation notebooks (data, EDA, preprocessing)
notebooks/07                ARCHIVED: the target-leakage case study
notebooks/08-12             the science: one executed experiment per regime
tests/                      57 pytest tests, run in CI on Python 3.10 and 3.12
```

Two usage modes, combinable because the API is a client of the library:
`from qsv import validate_state` in any Python project, or `uvicorn qsv.api:app`
over HTTP. A pedagogical web app (built separately) sits on top as a third layer.

In [1]:
# The package in three lines - the whole project distilled
from qsv import validate_state, preparation_qa

r = validate_state([0.6, 0.8, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0])
print(r.valid, "-", r.explanation)

r = validate_state([0.62, 0.8, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], n_shots=500)
print(r.valid, "-", r.explanation[:80], "...")

True - Exact amplitudes: validity is computed, not inferred. |norm^2 - 1| = 0.000e+00 vs tolerance 1e-06.
True - Finite-shot tomography (N=500, sigma=0.0224). Bias-corrected statistic |norm^2 - ...


## 5. Results

One line per regime, each backed by an executed notebook:

| Regime | Best approach | Key number | Notebook |
|---|---|---|---|
| Exact amplitudes | computation, not inference | leakage lesson | 07 (archived) |
| Stationary shot noise | 1-parameter threshold test | ties RF at every N | 08 |
| Common-mode correlated noise | threshold test (robust) | -0.6 pt at rho=0.9 | 09 |
| Anonymous cause diagnosis | ML, norm-dominated | Bayes limit: 40% scaling/noise confusion, proven by isotropy | 09 |
| Calibration drift | hybrid physics+ML | 0.967 vs 0.937 fixed threshold | 10 |
| Known-target preparation QA | (norm, fidelity) pair | 0.92 vs 0.667 per single statistic; rotated diagnosed at 100% | 11 |
| Scaling | d-agnostic; sizing curve | FPR <= 1% needs N ~ 2000 (margin 0.05, d=4) | 12 |

**Interpretation.** The through-line, learned four separate times, is that
*representation beats algorithm*: the logistic regression collapsing to 60% on a band
geometry, the RF drowning in 12 features while a GBM on (norm, time) learns the
calibration map, the depth-3 tree on the engineered pair beating the forest on raw
amplitudes. And the honest limits matter as much as the wins: the scaling/noise
confusion is a provable Bayes limit (isotropy argument), not a modelling failure, and
the ease of validation at higher dimension holds for our generator's invalid
population, not against adversarial states hugging the margin.

## 6. Hypotheses and limits

- **Gaussian additive noise** is a simplification of real tomography (which is
  multinomial per measurement setting); it is stated as such everywhere. Notably, a
  single-multinomial model would be norm-blind by construction - the additive model
  corresponds to per-component estimation from separate runs.
- **The invalid population is synthetic.** Margins, densities and therefore all
  accuracy numbers are properties of our generator. The sizing curve methodology
  transfers; the exact numbers do not.
- **The drift study leaks by design**: training and test share the drift realization,
  which models "learning a calibration map from history", not extrapolating to an
  unseen drift. Stated in notebook 10.
- **No entanglement, no mixed states, no density matrices**: pure states in a single
  computational basis. Milestone ideas beyond that are in the ROADMAP.

## 7. Error log - what actually went wrong, in my own words

*This section is my development logbook. Every entry corresponds to a real, documented
incident (session reports in `reports/`, git history). Time estimates are honest
approximations of what each mistake cost me.*

### 7.1 The big one: my model was grading its own answer key

**Context.** End of milestone 3, first model evaluation notebook (07). Random Forest,
proper train/val/test split, scaler fitted on train only - I was careful about all the
textbook leakage sources. Results: accuracy, precision, recall, AUC all exactly
1.0000, train-val gap 0.0000.

**What I thought first.** Honestly? For an evening, I was pleased. The features were
"physically motivated", the pipeline looked clean, I wrote "Prochaine étape: notebook
08 (Feature Importance)" and moved on. The perfect scores felt like confirmation, not
alarm.

**How I caught it.** The zeros bothered me more the next day. Zero errors out of 2000
test states never happens on a real statistical problem. I added four verification
cells at the end of the notebook; VERIFICATION 1 listed the training columns and
flagged `norm_deviation` as suspicious. I had excluded `norm_squared` from the
features "to avoid the trivial solution"... while keeping
`norm_deviation = |norm_squared - 1|`. Which is not close to the label. It IS the
label definition, passed through an absolute value.

**The failed fix.** My first instinct was to just drop `norm_deviation` and retrain.
Accuracy stayed suspiciously high, and working out why led to the deeper realization:
`entropy` and `purity` computed on raw |c_i|^2 also encode the norm (an invalid state
showed entropy = -96,088 in my own outputs - a red flag I had scrolled past), and even
the bare coordinates determine the norm exactly. On exact data there is NO feature set
that makes this a fair ML problem: it is either trivial or impossible.

**The actual solution.** Reformulate the problem physically: amplitudes observed
through finite-shot noise (sigma = 1/(2*sqrt(N))). Classes then genuinely overlap near
the boundary and deciding becomes statistics. Notebook 08 rebuilt the evaluation on
that basis; notebook 07 stays in the repo, archived with a warning cell, because the
mistake teaches more than the fix.

**What I learned.** Perfect metrics are an accusation, not a compliment. And "did I
exclude the leaky feature" is the wrong question - the right one is "is the label a
deterministic function of ANY subset of my features".

**Cost.** About two days end to end: the evening I believed it, half a day of
verification and dawning horror, a full day to understand the deterministic-problem
argument and rebuild on the noise model.

### 7.2 Six months of work living only on my hard drive

**Context.** Between December and July, I kept working - notebooks enriched, a whole
evaluation notebook written - but never committed. When I finally audited the repo,
about 920 real lines plus an entire notebook existed nowhere but my disk, and my
local view of the remote was six months stale (I had even edited the README directly
on GitHub without pulling, so my machine, my head and GitHub were three different
projects).

**How it surfaced.** A full git audit: `git status` showed a 2,271-line diff, of which
about 60% turned out to be pure CRLF/LF line-ending churn because the repo had no
`.gitattributes`. Untangling real changes from newline noise on Windows took longer
than committing them.

**Solution.** `.gitattributes` enforcing LF, a repaired `.gitignore` (a negation
pattern referenced `quantum_states_10k.csv` while the real file was
`quantum_states_10000.csv` - so the dataset I believed was versioned never was, and a
global ignore rule was silently hiding my whole `tests/` folder), then a series of
atomic commits with real messages, and tags on the milestones.

**What I learned.** Commit small and commit now; `git fetch` before comparing
anything; and on Windows, `.gitattributes` is not optional. Also: an ignore rule you
wrote in thirty seconds can hide a test suite for six months.

**Cost.** A focused half-day to clean up; unmeasurable risk carried for six months.

### 7.3 The tolerance that lied by 10x

**Context.** My normalization check used `np.allclose(norms, 1.0, atol=tolerance)`.

**The error.** `np.allclose` keeps its default `rtol=1e-5` unless you zero it, and the
criterion is `|x - 1| <= atol + rtol*|1|`. My advertised tolerance of 1e-6 was
actually about 1.1e-5 - an order of magnitude looser than documented. For a project
whose entire subject is "how close to 1 is valid", the validator itself was lying
about its threshold.

**How I found it.** Not by a bug - by reading the NumPy docs during the code-quality
audit and doing the arithmetic. A test now pins the strict behaviour: an offset of
2e-5 must be REFUSED at tolerance 1e-6 (the old code accepted it).

**Lesson.** Read the default arguments of library functions you put in a definition.
Especially the ones that define your product.

**Cost.** Twenty minutes to fix, once seen. Potentially poisonous forever if unseen.

### 7.4 Two hypotheses the data refused

**Correlated noise (notebook 09).** I was convinced common-mode noise (carrier
vibration, shared readout laser) would break the threshold test and let the Random
Forest shine. Measured: the test loses 0.6 points between rho=0 and rho=0.9 and the
forest gains nothing. The sum over 2d components acts as a natural filter of the
common mode - obvious in hindsight, wrong in foresight. I published the negative
result: for this regime, an embedded validator needs no learned model, and that is
worth knowing.

**Drift + time feature (notebook 10).** Second conviction: give the forest the
acquisition time and it will learn the calibration drift. Measured: RF with 12
features + time, 0.939 - barely above the collapsed fixed threshold - while a
zero-learning rolling median hit 0.964. The interaction I wanted it to find was
drowned in irrelevant features; the same information given to a GBM as (norm, time)
ALONE reached 0.962. Third time the same lesson hit me: representation beats
algorithm. The winner was the hybrid (physics statistic as a feature): 0.967.

**Cost.** None in time - the experiments-first rule meant no text had to be retracted.
Some in ego, which is the point of the rule.

### 7.5 Assorted smaller scars

- **My notebook called functions that no longer existed.** Notebook 06 used
  `create_logistic_regression` and `train_model` from an old version of
  `preprocessing.py` - it could not have run against the current module. Classic
  notebook/module drift; fixed by rewiring it to the real API and re-executing
  everything end to end. Lesson: a notebook that is not re-executed is documentation
  of a past that may no longer exist. (~1 h)
- **numpy.bool broke my API.** First run of the FastAPI tests: `sigma_from_shots`
  returns np.float64, comparisons produce numpy booleans, and FastAPI cannot
  JSON-serialize them. Explicit float()/bool() casts at the API boundary. Unit tests
  of the physics could never catch this - integration tests earned their keep on day
  one. (~15 min)
- **sed silently matched nothing.** Editing `.gitignore` with a `$`-anchored sed
  pattern failed because the lines ended in invisible CRLF. The command "succeeded",
  the file was unchanged, and the bug resurfaced downstream. Exactly the class of
  problem `.gitattributes` had just been added to prevent. (~30 min)
- **A dev environment died mid-session** and took uncommitted work with it. Rebuilding
  from the repo and re-running produced bit-identical numbers thanks to seeded
  generators. The half-day I once spent making everything reproducible repaid itself
  in about ten minutes. (recovery: ~15 min)

## 8. Conclusion and future work

The project set out to train a classifier and ended up somewhere better: a validated,
packaged, documented answer to the question "when does ML earn its place in a
physics-defined decision problem?" - with the honest answer being "less often than I
assumed, and best when married to the physics".

What I would do next, roughly in order:

1. **Multinomial tomography model** - replace the Gaussian simplification with
   per-setting multinomial sampling and see which conclusions survive.
2. **Adversarial invalid states** - populations hugging the margin, to stress the
   sizing curve beyond our generator's easy cases.
3. **Mixed states and density matrices** - validity becomes positivity +
   unit trace; richer geometry, richer failure modes.
4. **Out-of-distribution drift** - test the drift models on unseen drift
   realizations, where the learned calibration map must extrapolate.
5. **The web app as a teaching platform** - guided paths and adaptive quizzes on top
   of the playgrounds.

The repository stands at: 57 green tests in CI, 13 notebooks (12 executed + 1
archived as a case study), a pip-installable package, a REST API, and this report.